## 准备数据

In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [2]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [3]:
class myModel:
    def __init__(self):
        # 初始化权重和偏置
        self.W1 = tf.Variable(tf.random.normal([784, 128], stddev=0.1), dtype=tf.float32)
        self.b1 = tf.Variable(tf.zeros([128]), dtype=tf.float32)
        self.W2 = tf.Variable(tf.random.normal([128, 10], stddev=0.1), dtype=tf.float32)
        self.b2 = tf.Variable(tf.zeros([10]), dtype=tf.float32)

    def __call__(self, x):
        # 将输入展平为 (batch_size, 784)
        x = tf.reshape(x, [-1, 784])
        # 第一层：线性变换 + ReLU 激活
        h = tf.nn.relu(tf.matmul(x, self.W1) + self.b1)
        # 第二层：线性变换，输出 logits
        logits = tf.matmul(h, self.W2) + self.b2
        return logits
        
model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [4]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    for g, v in zip(grads, trainable_vars):
        v.assign_sub(0.01*g)

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [5]:
train_data, test_data = mnist_dataset()
for epoch in range(50):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 2.6203692 ; accuracy 0.05015
epoch 1 : loss 2.5924084 ; accuracy 0.050616667
epoch 2 : loss 2.5669124 ; accuracy 0.051733334
epoch 3 : loss 2.543468 ; accuracy 0.053166665
epoch 4 : loss 2.5217538 ; accuracy 0.055483334
epoch 5 : loss 2.501511 ; accuracy 0.057366665
epoch 6 : loss 2.4825325 ; accuracy 0.0596
epoch 7 : loss 2.4646516 ; accuracy 0.062366668
epoch 8 : loss 2.4477265 ; accuracy 0.06535
epoch 9 : loss 2.4316444 ; accuracy 0.06908333
epoch 10 : loss 2.4163108 ; accuracy 0.07286666
epoch 11 : loss 2.4016404 ; accuracy 0.07688333
epoch 12 : loss 2.3875654 ; accuracy 0.080633335
epoch 13 : loss 2.374026 ; accuracy 0.084633335
epoch 14 : loss 2.3609693 ; accuracy 0.0891
epoch 15 : loss 2.3483503 ; accuracy 0.093666665
epoch 16 : loss 2.3361287 ; accuracy 0.09798333
epoch 17 : loss 2.3242714 ; accuracy 0.10271667
epoch 18 : loss 2.3127468 ; accuracy 0.10853333
epoch 19 : loss 2.3015282 ; accuracy 0.114033334
epoch 20 : loss 2.2905915 ; accuracy 0.11873333
epoch 21 